<div style="border-left:5px solid #2a78d6;padding:6px 0 6px 20px;margin-bottom:6px">

# AeroTrack-Net — Training
## Review 2 · from a verified corpus to a trained detector

</div>

> ### Review 1 measured 723,675 boxes and let the measurements pick the architecture.
> ### This notebook trains it — and measures the one number that tests the claim: **AP on 9-pixel targets**.

---

**How to run:** set the config in the next cell, then *Run All*. Everything below is
idempotent and ordered — corpus preparation, environment gate, pre-flight, then the
launch. Nothing needs babysitting between cells.

**Run All finishes in minutes, not hours.** The training itself is handed to a *detached*
process (`BACKGROUND = True`) and this notebook returns immediately. The run survives a
closed notebook, a restarted kernel and a closed terminal; it holds a budgeted share of
RAM and VRAM so the machine stays usable; and it writes a checkpoint every epoch, so
stopping it costs at most one epoch. Come back to the monitor cell in Act IV whenever you
want to know how it is doing.

**Kernel:** the project venv — `venv\Scripts\python.exe`. It carries the **cu130** CUDA
build of PyTorch; a CPU wheel, or a wheel without `sm_120` kernels, is rejected by the
environment gate below before a single byte is allocated.

---

### What this run does differently from the Review-1 staging code

| # | Change | Why (measured, from `a_inspection/`) |
|---|---|---|
| 1 | **Sequence-level splits** | Train and val were the *same frames*. Consecutive frames differ by a median of 5.10 px (Anti-UAV) / 0.64 px (CST) — even a random frame split leaks. Now a hard assertion blocks the run. |
| 2 | **Letterbox, not stretch** | Stretching 640×512 and 1920×1080 to a square gave the *same physical drone* two different aspect ratios (1.33 vs 0.93). The network was being asked to unlearn its own preprocessing. |
| 3 | **Iteration-budgeted epochs** | An epoch was 394k samples ≈ hours. It is now a fixed budget of weighted draws — 15× more LR steps, checkpoints and validation signal per hour. |
| 4 | **Temporally-consistent augmentation** | There was none. One transform is now sampled per *sample* and applied to t−1, t, t+1 identically; anything else injects motion that never happened. No mosaic, no mixup, no vertical flip. |
| 5 | **Partial COCO transfer** | Only the 5 SPDConv stems and the head are novel; the rest is stock YOLO11n and starts from COCO features instead of noise. |
| 6 | **Size-stratified evaluation** | Aggregate mAP is dominated by easy large targets. `micro` AP (<0.03 % of frame) is the number that tests SPD-Conv, so model selection optimises it explicitly. |
| 7 | **bf16, not fp16** | CIoU on degenerate ~9 px boxes goes NaN in fp16. bf16 has fp32's exponent range and is native on Blackwell. |

---
## 0 · Configuration — the only cell you edit

In [1]:
# ============================ RUN CONFIGURATION ============================ #
# Stages run in the order listed. Each is a full, independent job with its own
# run directory under runs/. See 00_TRAINING_ROADMAP.md 6.1.
#
#   "overfit"  S0  1 fixed batch, 200 iters   ~2 min   proves the loss can learn
#   "smoke"    S1  2,000 samples, 3 epochs    ~10 min  proves the plumbing holds
#   "pilot"    S2  CST only, 20 epochs        ~1 h     hardest data, fastest signal
#   "baseline" S3  full corpus, budgeted      hours    THE model
RUN_STAGES = ["overfit", "smoke", "baseline"]

EPOCHS            = 150        # baseline epochs
SAMPLES_PER_EPOCH = 25_000     # iteration budget per epoch (roadmap 3.2)
BATCH             = 32         # 32 = MAX SAFE VRAM limit on RTX 5050 (4.78 GB / 8 GB)
WORKERS           = 4          # 4 = MAX SAFE limit keeping >=6GB RAM buffer
IMGSZ             = 640
LR0               = 1e-3
PATIENCE          = 30         # validations without improvement before stopping

# ===================== SHARING THE MACHINE WHILE IT TRAINS ================= #
# 23.6 GB of RAM and one 8 GB card, and the plan is to keep using both -- three
# slide decks and a browser -- for the whole run. Four knobs decide whether that
# works. They are the only things in this notebook that are about the MACHINE
# rather than about the model.

BACKGROUND = True       # Run training in a DETACHED process instead of inside
                        # this kernel. Nothing else here comes close to
                        # mattering as much: the run stops being a child of
                        # Jupyter, so it survives a closed notebook, a restarted
                        # kernel, a closed terminal and a closed VS Code -- and
                        # the kernel's own multi-GB footprint is handed back
                        # before the long run even starts.

RESERVE_RAM_GB = 6.0    # Host RAM training may NOT plan for: your decks, your
                        # browser, and room for the OS to grow. The worker count
                        # is derived from what is left, so THIS is the knob to
                        # tune, not WORKERS. Each DataLoader worker is a fresh
                        # Windows interpreter: ~2.4 GB of commit, ~1.3 GB
                        # resident, because spawn re-imports numpy and torch per
                        # worker. Raise this if the machine feels slow; lower it
                        # if the GPU is being starved (watch img/s in the log).

RESERVE_VRAM_GB = 2.0   # GPU memory left for the desktop. The card is not ours
                        # alone -- the compositor, the browser and PowerPoint's
                        # renderer hold ~1 GB before a slideshow even starts,
                        # and a slideshow costs several hundred MB more. The
                        # batch probe budgets against FREE VRAM minus this, and
                        # a transient OOM now costs one batch, not the run.

BG_PRIORITY = "normal"  # Maximum CPU priority for fastest data loading
CPU_THREADS = 4         # torch intra-op threads.
BG_TAG = "train"        # names the background run: runs/_bg/<tag>.json

# Evaluation
EVAL_AFTER_TRAINING = True     # full size-stratified suite on the val split
EVAL_CAP_PER_DATASET = 8000    # 0 = every val frame (slow but complete)

# Ablations -- each defends one design decision. Run after the baseline exists.
#   "no_spd" | "single_frame" | "no_sampler" | "no_letterbox" | "scratch"
#   | "p2_head" | "no_augment"
RUN_ABLATIONS       = []
ABLATION_EPOCHS     = 40

# The test split -- cst[test] especially -- is the project's only honest
# cross-domain measurement. Spend it ONCE, on the final model. The cell that
# uses it refuses to run while training is still in flight, so leaving this True
# is safe: it takes effect the first time you re-run Act VII after the
# background run has finished.
RUN_FINAL_TEST = True

# Augmentation strengths (applied identically to t-1, t, t+1)
AUGMENT = dict(
    hflip=0.5,              # safe; vertical flip is NOT used (sky-above-ground)
    scale=(0.90, 1.25),     # biased above 1.0: never shrink an already-micro target
    translate=0.10,         # attacks Anti-UAV's 90% centre bias directly
    brightness=0.15,        # thermal crossover is a real IR failure mode
    contrast=0.15,
    copy_paste=0.30,        # paste a micro target, with its motion, elsewhere
    copy_paste_max=3,
)

# Hardness weights for the sampler (b_pipeline/imbalance_sampler.py).
# Raise "bird" if the evaluation shows bird false positives; raise "micro" if
# micro-AP stays flat near zero.
SAMPLER_WEIGHTS = dict(bird=8.0, micro=6.0, small=3.0, standard=1.0, empty=0.5)

# Temporal gap g: the stack is (t-g, t, t+g). g=1 is the Review-1 design.
# CST moves a median of 0.64 px/frame, so g=2..3 makes motion genuinely visible --
# worth an ablation once the baseline exists.
TEMPORAL_GAP = 1

RESUME_FROM = None             # e.g. "runs/aerotrack_spd_v1/weights/last.pt"
# =========================================================================== #
print("config loaded")

config loaded


In [2]:
import os, sys, json, gc, time, subprocess, warnings
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import HTML, display

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "c_model").is_dir() and (ROOT / "b_pipeline").is_dir():
        break
    ROOT = ROOT.parent
else:
    raise SystemExit("Run this notebook from the project root (it must contain c_model/).")

sys.path.insert(0, str(ROOT / "c_model"))
sys.path.insert(0, str(ROOT / "b_pipeline"))
UNIFIED = ROOT / "b_pipeline" / "data_unified"
RUNS = ROOT / "runs"
PY = sys.executable

# ---- the Review-1 visual identity, so the two decks read as one document ---- #
INK, MUTED, SURF, CARD, LINE, ACCENT = "#0f1114", "#3f444c", "#fcfcfb", "#f1f1ec", "#c4c4be", "#a3271a"
DS_COLOR = {"anti_uav": "#2a78d6", "cst": "#008300", "det_fly": "#eda100"}
# Darkened variants for TEXT. Raw #eda100 amber is ~1.9:1 on white — fine as a bar,
# unreadable as a number. Same trick as rev1.ipynb, same reason.
DS_INK = {"anti_uav": "#17549e", "cst": "#00631a", "det_fly": "#7d5000"}
_SAFE = {DS_COLOR[k]: DS_INK[k] for k in DS_COLOR}
mpl.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "figure.dpi": 110, "font.size": 11, "axes.edgecolor": "#b8b8b2",
    "axes.labelcolor": INK, "text.color": INK, "xtick.color": MUTED,
    "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#dededa", "grid.linewidth": .8,
    "legend.frameon": False,
})
_BOX = f"background:{SURF};border:1px solid {LINE};border-radius:10px;"

def title(text, kicker=None):
    k = (f"<div style='font-size:11.5px;letter-spacing:.10em;text-transform:uppercase;"
         f"font-weight:700;color:{ACCENT};margin-bottom:5px'>{kicker}</div>") if kicker else ""
    display(HTML(f"<div style='{_BOX}padding:13px 18px;margin:22px 0 8px;border-left:5px solid {INK}'>"
                 f"{k}<div style='font-size:18px;font-weight:800;color:{INK};line-height:1.3'>{text}</div></div>"))

def hero(items, caption=None):
    cells = "".join(
        f"<div style='flex:1;min-width:150px;padding:14px 18px'>"
        f"<div style='font-size:29px;font-weight:800;color:{_SAFE.get(c, c)};line-height:1.05'>{v}</div>"
        f"<div style='font-size:12px;color:{MUTED};margin-top:7px;line-height:1.4'>{l}</div></div>"
        for v, l, c in items)
    cap = (f"<div style='font-size:12px;color:{MUTED};padding:2px 18px 13px'>{caption}</div>") if caption else ""
    display(HTML(f"<div style='{_BOX}padding:4px 0;margin:10px 0'>"
                 f"<div style='display:flex;flex-wrap:wrap'>{cells}</div>{cap}</div>"))

def panel(html, accent=None):
    display(HTML(f"<div style='background:{SURF};border:1px solid {LINE};"
                 f"border-left:4px solid {accent or INK};border-radius:0 8px 8px 0;"
                 f"padding:14px 18px;margin:10px 0 16px;font-size:13px;line-height:1.7;"
                 f"color:{INK}'>{html}</div>"))

def table(df, note=None):
    sty = (df.style.hide(axis="index").set_table_styles([
        {"selector": "th", "props": [("background", CARD), ("font-size", "12px"),
                                     ("text-align", "left"), ("padding", "8px 12px"),
                                     ("border-bottom", f"2px solid {LINE}"),
                                     ("color", INK), ("font-weight", "700")]},
        {"selector": "td", "props": [("padding", "7px 12px"), ("font-size", "13px"),
                                     ("background", SURF), ("color", INK),
                                     ("border-bottom", "1px solid #e4e4de")]},
        {"selector": "", "props": [("border-collapse", "collapse"),
                                   ("border", f"1px solid {LINE}"),
                                   ("border-radius", "8px"), ("overflow", "hidden")]}]))
    display(sty)
    if note:
        display(HTML(f"<div style='background:{CARD};border:1px solid {LINE};border-radius:8px;"
                     f"padding:11px 15px;font-size:12.5px;color:{MUTED};margin:6px 0 16px;"
                     f"line-height:1.65'>{note}</div>"))

# --------------------------------------------------------------------------- #
#  Module hygiene -- the fix for "'TrainConfig' object has no attribute ..."
# --------------------------------------------------------------------------- #
_PROJECT_MODULES = {
    "train", "preflight", "evaluation", "aerotrack_trainer", "stacked_dataset",
    "imbalance_sampler", "custom_modules", "bytetrack", "lstm_forecaster",
    "track_eval", "inference", "val", "selfcheck",
}

def purge_project_modules(verbose=False):
    """Drop every project module from sys.modules so the next import is fresh.

    `importlib.reload(train)` is not enough here, and it fails in a way that
    wastes an afternoon. Reloading train.py re-executes
    `from aerotrack_trainer import TrainConfig`, and that binds whatever is
    ALREADY in sys.modules -- the OLD class object. New caller, old callee, and
    the symptom is an AttributeError naming a field that plainly exists on disk:

        AttributeError: 'TrainConfig' object has no attribute 'val_workers'

    Purging the whole set is the only version of this that is correct, because
    the seven project modules import each other. Call it before the first
    project import in a session, and again after editing any project .py.
    """
    dropped = [m for m in list(sys.modules)
               if m in _PROJECT_MODULES or m.startswith(("c_model.", "b_pipeline."))]
    for m in dropped:
        sys.modules.pop(m, None)
    if verbose and dropped:
        print(f"[purge] dropped {len(dropped)} project modules: {sorted(dropped)}")
    return dropped


print("project root :", ROOT)
print("interpreter  :", PY)

project root : a:\AEROTRACK_OMEN
interpreter  : a:\AEROTRACK_OMEN\venv\Scripts\python.exe


---
<div style="background:#f2f6fc;border-left:5px solid #2a78d6;padding:14px 20px;border-radius:0 8px 8px 0;color:#0f1114">

# ACT I · The Gate
### Blackwell, or nothing

</div>

The RTX 5050 is Blackwell — compute capability **sm_120**. A PyTorch wheel built before
Blackwell support installs cleanly, reports `cuda.is_available() == True`, and then dies at
the first real kernel launch with *"no kernel image is available for execution on the
device"*. `is_available()` is **not** the test. `sm_120 in torch.cuda.get_arch_list()` is,
and a matmul that actually runs is the proof.

In [3]:
import torch, cv2, platform, psutil, shutil

purge_project_modules()      # one consistent load of every project module
import preflight as P

title("Environment gate", kicker="Act I · hardware")

res = P.check_environment(require_sm120=True)
print(res)

cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
vram = (torch.cuda.get_device_properties(0).total_memory / 1024**3) if torch.cuda.is_available() else 0
free_gb = shutil.disk_usage(str(ROOT)).free / 1024**3

hero([
    (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU",
     f"compute capability {cap[0]}.{cap[1]} &mdash; <b>sm_{cap[0]}{cap[1]}</b>", DS_COLOR["anti_uav"]),
    (f"{vram:.1f} GB", "VRAM &mdash; the activation budget", DS_COLOR["cst"]),
    (torch.__version__, f"torch &middot; CUDA {torch.version.cuda}", MUTED),
    (f"{psutil.cpu_count(logical=False)}C/{psutil.cpu_count()}T",
     f"CPU &middot; {psutil.virtual_memory().total/1024**3:.0f} GB RAM", MUTED),
    (f"{free_gb:.0f} GB", f"free on {str(ROOT)[:2]}", MUTED),
], caption=f"OpenCV {cv2.__version__} &middot; {platform.platform()}")

# Host memory matters as much as VRAM here: a 9-channel 640x640 uint8 sample is
# 3.69 MB, so the loader queues are measured in GB, and on Windows it is COMMIT
# (RAM + pagefile) that runs out first. When commit is exhausted the OS kills a
# DataLoader worker, which surfaces only as "worker exited unexpectedly".
vm = psutil.virtual_memory()
sw = psutil.swap_memory()
commit_free = (vm.available + max(sw.total - sw.used, 0)) / 1024**3
_lo = commit_free < 8
print(f"[host] RAM {vm.total/1024**3:.1f} GB total, {vm.available/1024**3:.1f} GB available "
      f"| commit headroom ~{commit_free:.1f} GB")
if _lo:
    panel(f"<b>Only ~{commit_free:.1f} GB of commit headroom.</b> This notebook's loaders "
          f"need a few GB of it. If a previous run is still resident in this kernel, "
          f"<b>restart the kernel now</b> &mdash; otherwise the OS will kill a DataLoader "
          f"worker mid-epoch and PyTorch will report it as "
          f"<code>worker (pid(s) ...) exited unexpectedly</code>, which names neither the "
          f"cause nor the fix.", ACCENT)

if not res.ok:
    raise SystemExit("Environment gate failed — fix the wheel before continuing. "
                     "See the FAIL detail printed above.")
if torch.cuda.is_bf16_supported():
    panel("<b>bf16 is available and will be used.</b> fp16 mixed precision goes NaN on "
          "CIoU of degenerate boxes, and our median target is a ~9&times;9 px box &mdash; "
          "the degenerate case is the <i>normal</i> case here. bf16 carries fp32's exponent "
          "range at fp16's bandwidth and is native on Blackwell.", "#0f7a2e")

# ---- the budget this run will be held to ---------------------------------- #
title("The memory budget", kicker="Act I &middot; sharing the machine")

_free_vram = (torch.cuda.mem_get_info()[0] / 1024**3) if torch.cuda.is_available() else 0
_used_vram = vram - _free_vram
hero([
    (f"{vm.available/1024**3:.1f} GB", "host RAM free right now", DS_COLOR["anti_uav"]),
    (f"{RESERVE_RAM_GB:.1f} GB", "reserved for <b>your</b> decks and browser", ACCENT),
    (f"{_used_vram:.1f} GB", "VRAM the desktop already holds", DS_COLOR["det_fly"]),
    (f"{RESERVE_VRAM_GB:.1f} GB", "VRAM reserved for it to grow into", ACCENT),
], caption=f"Commit headroom ~{commit_free:.1f} GB (RAM + pagefile). Worker count and "
           f"batch size are both <b>derived</b> from these four numbers, not typed in.")

panel(f"<b>Why the reserve is a knob and not a constant.</b> A DataLoader worker on "
      f"Windows is a whole fresh interpreter -- spawn re-imports numpy and torch into "
      f"each one -- so it costs ~2.4&nbsp;GB of commit and ~1.3&nbsp;GB resident "
      f"<i>before</i> it queues a single batch. With {RESERVE_RAM_GB:.0f}&nbsp;GB held "
      f"back, <code>train.plan_workers()</code> will authorise roughly "
      f"<b>{max(int((vm.available/1024**3 - RESERVE_RAM_GB - 1.5) // 1.3), 0)} workers</b> "
      f"out of the ceiling of {WORKERS}. Fewer workers is a slower epoch, never a dead "
      f"run -- and the loader only has to stay ahead of ~72&nbsp;img/s, which is what "
      f"this GPU sustains on a 9-channel batch.", DS_COLOR["anti_uav"])


[PASS] E0 environment: NVIDIA GeForce RTX 5050 Laptop GPU cap=(12, 0) (sm_120) | torch 2.13.0+cu130 cuda 13.0 | arch_list=['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120'] | matmul finite=True | VRAM 8.0 GB (6.8 free) | bf16=True


[host] RAM 23.6 GB total, 9.2 GB available | commit headroom ~21.1 GB


---
<div style="background:#f2fbf3;border-left:5px solid #008300;padding:14px 20px;border-radius:0 8px 8px 0;color:#0f1114">

# ACT II · The Corpus
### Prepared once, asserted every time

</div>

Three artifacts must exist before epoch 1, and none of them may be built *inside* it:

* **`label_cache.npz`** — every YOLO label parsed once into memory-resident arrays. The
  sampler's hardness scan is otherwise a 394k-file disk walk that happens *during* the
  first epoch: 30+ minutes of dead GPU time, every run.
* **`splits.json`** — the sequence-level train/val/test map, including the CST validation
  holdout carved out with a fixed seed.
* **`sample_hardness.csv`** — per-frame hardness category for the weighted sampler.

The cell below builds them if they are missing and is a no-op if they are not.

In [4]:
title("Corpus preparation", kicker="Act II · phase B")

MANIFEST = UNIFIED / "manifest.csv"
if not MANIFEST.exists():
    raise SystemExit(
        "manifest.csv is missing. Build the corpus first:\n"
        f"  {PY} a_inspection/1_uncompress_datasets.py\n"
        f"  {PY} b_pipeline/1_standardize_annotations.py --full-antiuav")

need = [p for p in (UNIFIED / "label_cache.npz", UNIFIED / "splits.json") if not p.exists()]
if need:
    print(f"missing {[p.name for p in need]} — running 3_prepare_training.py "
          f"(one-time; a few minutes)\n")
    r = subprocess.run([PY, str(ROOT / "b_pipeline" / "3_prepare_training.py")],
                       cwd=str(ROOT))
    if r.returncode != 0:
        raise SystemExit("3_prepare_training.py failed — see the output above.")
else:
    print("label_cache.npz + splits.json present — skipping preparation.")

spec = json.loads((UNIFIED / "splits.json").read_text(encoding="utf-8"))
summ = json.loads((UNIFIED / "standardization_summary.json").read_text(encoding="utf-8"))

label_cache.npz + splits.json present — skipping preparation.


In [5]:
import pandas as pd

title("The split — sequence-level, and disjoint by construction", kicker="Act II · splits")

rows = []
for phase in ("train", "val", "test"):
    for dk, c in spec["counts"][phase].items():
        rows.append([phase, dk, f"{c['sequences']:,}", f"{c['frames']:,}",
                     f"{c['materialized']:,}"])
table(pd.DataFrame(rows, columns=["Phase", "Dataset", "Sequences", "Frames", "Images on disk"]),
      note="<b>CST ships no validation split</b>, so "
           f"{len(spec['cst_val_holdout_seqs'])} of its training sequences are held out with a "
           f"fixed seed ({spec['seed']}) and written to <code>splits.json</code>. "
           "<code>cst[test]</code> is not touched by anything in this notebook unless "
           "<code>RUN_FINAL_TEST</code> is set.")

tot_tr = sum(c["materialized"] for c in spec["counts"]["train"].values())
tot_va = sum(c["materialized"] for c in spec["counts"]["val"].values())
tot_te = sum(c["materialized"] for c in spec["counts"]["test"].values())
hero([
    (f"{tot_tr:,}", "training images (sequence-disjoint from val)", DS_COLOR["anti_uav"]),
    (f"{tot_va:,}", "validation images", DS_COLOR["cst"]),
    (f"{tot_te:,}", "test images &mdash; <b>sealed</b>", ACCENT),
    (f"{len(spec['cst_val_holdout_seqs'])}", "CST sequences held out for validation", MUTED),
])

panel("<b>Why this is the highest-priority fix in the whole project.</b> The Review-1 "
      "trainer filtered by <i>dataset</i> and never by <i>split</i> &mdash; the validation set "
      "was a strict subset of the training set. Every val frame had been trained on, so mAP "
      "would have come back near-perfect and meaningless. Worse, for video even a random "
      "<i>frame-level</i> split leaks: at a median 0.64 px/frame (CST), frame <i>t</i> in train "
      "and frame <i>t+1</i> in val are visually the same image. Only a sequence-level split is "
      "honest, and the assertion below enforces it on every single run.", ACCENT)

Phase,Dataset,Sequences,Frames,Images on disk
train,anti_uav,320,"299,056","299,056"
train,cst,69,"75,005","75,005"
train,det_fly,"4,698","4,698","4,698"
val,anti_uav,134,"123,998","123,998"
val,cst,15,"15,007","15,007"
val,det_fly,"1,565","1,565","1,565"
test,anti_uav,182,"170,748","170,748"
test,cst,60,"72,175","72,175"
test,det_fly,"1,567","1,567","1,567"


---
<div style="background:#fffaf0;border-left:5px solid #eda100;padding:14px 20px;border-radius:0 8px 8px 0;color:#0f1114">

# ACT III · Assembly & Pre-flight
### Eight tests, one minute, every launch

</div>

A 20-hour run that ends in an unloadable checkpoint is a 20-hour loss. So is a run whose
validation set was in its training set. Both failure modes are cheap to detect *before*
the run and impossible to fix after it.

In [6]:
purge_project_modules()          # see the note in the setup cell: a stale
                                 # aerotrack_trainer is what produces
                                 # "'TrainConfig' has no attribute 'val_workers'"
import train as T
import preflight as P

title("Build the job", kicker="Act III &middot; assembly")

# The pre-flight is run against the job the LONG run will actually use -- the
# full corpus, its real sequence-level split, its real weighted sampler -- and
# not against S0's sixteen fixed images. Checking S0 would make T4 (leakage)
# inapplicable and would draw the hardness chart below over a 16-sample corpus,
# which is worse than not drawing it. S0 still runs; it runs inside the
# background job, where a failure aborts the whole staircase before S1 starts.
PREFLIGHT_STAGE = "baseline" if "baseline" in RUN_STAGES else (
    [s for s in RUN_STAGES if s != "overfit"] + ["baseline"])[0]

# Every worker-count decision downstream reads this, including the detached
# process launched later -- it is passed through the environment so the
# background run inherits exactly the budget set in the config cell.
os.environ["AEROTRACK_RESERVE_GB"] = str(RESERVE_RAM_GB)

device = T.require_gpu()
T.limit_cpu_threads(CPU_THREADS)

over = dict(workers=WORKERS, imgsz=IMGSZ, lr0=LR0, temporal_gap=TEMPORAL_GAP)
if BATCH:
    over["batch"] = BATCH

trainer, ctx = T.build(
    PREFLIGHT_STAGE,
    device=device,
    overrides=over,
    augment_kwargs=AUGMENT,
    sampler_weights=SAMPLER_WEIGHTS,
)

hero([
    (f"{ctx['n_params']/1e6:.2f} M", "parameters", DS_COLOR["anti_uav"]),
    (f"{ctx['n_spdconv']}", "SPDConv layers replacing strided Conv", "#0f7a2e"),
    (f"{ctx['transferred'][0]:,}/{ctx['transferred'][1]:,}",
     "tensors transferred from COCO yolo11n", DS_COLOR["cst"]),
    ("[B, 9, 640, 640]", "input tensor &mdash; t&minus;1, t, t+1", MUTED),
])


[cpu] torch intra-op threads -> 4
[corpus] train 378,759 samples {'anti_uav': 299056, 'cst': 75005, 'det_fly': 4698}
[corpus] val   2,000 samples {'anti_uav': 1000, 'cst': 700, 'det_fly': 300}
[corpus] train sequences 5,087 | val sequences 448 | disjoint OK
[hardness] vectorised from label cache: 378,759 samples
[sampler] corpus category mix (uniform) : {'bird': '0.08%', 'micro': '9.71%', 'small': '80.91%', 'standard': '3.94%', 'empty': '5.36%'}
[sampler] expected sampled mix (weighted): {'bird': '0.20%', 'micro': '18.91%', 'small': '78.75%', 'standard': '1.28%', 'empty': '0.87%'}
[commit] 14.3 GB free commit, reserve 8.5 GB, ~2.4 GB per worker -> 2 workers
[ram]    8.1 GB free RAM, reserve 7.5 GB, ~1.3 GB per worker -> 0 workers
[plan]   using 0 workers  <- CAPPED from 4
[plan]   4 workers would need ~18.1 GB commit / ~12.7 GB RAM. Lower AEROTRACK_RESERVE_GB, or close other apps, to use more.
[plan]   falling back to workers=0 (in-process loading). Slow but it cannot be killed by the 

In [7]:
title("Measure the real VRAM ceiling", kicker="Act III &middot; batch size")

if not BATCH:
    best_b, probe = P.probe_batch_size(
        trainer.model, trainer.criterion, device, imgsz=IMGSZ, ch=9,
        candidates=(4, 8, 12, 16, 24, 32), headroom_gb=RESERVE_VRAM_GB,
        amp_dtype=trainer.amp_dtype)
    if best_b != ctx["cfg"].batch:
        print(f"\nrebuilding with the measured batch ({ctx['cfg'].batch} -> {best_b})")
        T.release(trainer, ctx)
        del trainer, ctx; gc.collect(); torch.cuda.empty_cache()
        over["batch"] = best_b
        trainer, ctx = T.build(PREFLIGHT_STAGE, device=device, overrides=over,
                               augment_kwargs=AUGMENT,
                               sampler_weights=SAMPLER_WEIGHTS)
    BATCH = best_b

panel(f"Ultralytics' autobatch would mis-measure this network twice over: the input "
      f"tensor is 3&times; fatter than RGB, and <b>SPD-Conv is <i>more</i> "
      f"activation-hungry than strided convolution in the early layers</b> &mdash; the "
      f"very first block holds <code>B&times;36&times;320&times;320</code>, because "
      f"nothing was thrown away. So we measure a real forward+backward instead of "
      f"estimating one. Gradient accumulation then restores the effective batch to "
      f"<b>{ctx['cfg'].nbs}</b> without risking an OOM twenty hours in.<br><br>"
      f"<b>The headroom is {RESERVE_VRAM_GB} GB, not the usual 1.5.</b> This card also "
      f"draws your desktop. A slideshow entering presenter mode, or a video in a "
      f"browser tab, takes VRAM without asking, and the batch that fits an idle "
      f"desktop is not the batch that fits a working one.",
      DS_COLOR["anti_uav"])


In [8]:
title("Pre-flight &mdash; T1 through T8", kicker="Act III &middot; the checks")

checks = [
    P.check_dataset(ctx["val_ds"], batch_size=min(4, ctx["cfg"].batch), imgsz=IMGSZ),
    P.check_boundary(ctx["val_ds"]),
    P.check_leakage(ctx["train_seq_keys"], ctx["val_seq_keys"]),
    P.check_collate(trainer, ctx["val_ds"], batch_size=min(4, ctx["cfg"].batch)),
    P.check_checkpoint_roundtrip(trainer, device, imgsz=IMGSZ),
    P.check_throughput(ctx["train_loader"], n_batches=20),
    P.check_vram(trainer, ctx["train_ds"], ctx["cfg"].batch, device),
]
# S0 deliberately validates on its own training batch — T4 cannot apply there.
allow = ("T4",) if ctx["stage_cfg"].get("val_from_train") else ()
PREFLIGHT_OK = P.run_all(checks, raise_on_fail=True, allow_fail=allow)


KeyboardInterrupt: 

### Look at what the network actually receives

Every number above can be right while the tensor is still wrong. This renders one real
training sample — letterboxed, augmented, with its ground-truth box carried through the
same transform — straight out of the loader that is about to train the model.

In [ ]:
from stacked_dataset import AeroTrackDataset, TemporalAugment

title("One real 9-channel training sample", kicker="Act III · visual proof")

# find a val sample that actually carries a target
vidx = ctx["val_ds"].index
counts = vidx.box_counts()
with_target = np.flatnonzero(counts > 0)
if not len(with_target):
    raise SystemExit("No validation frame carries a ground-truth box — the label "
                     "pipeline is broken. Check b_pipeline/1_standardize_annotations.py "
                     "(a missing Pillow silently writes every CST label empty).")
pick = int(with_target[len(with_target) // 2])

clean = AeroTrackDataset(vidx, img_size=(IMGSZ, IMGSZ), augment=None,
                         letterbox=True, out_dtype="uint8", return_meta=True)
aug = AeroTrackDataset(vidx, img_size=(IMGSZ, IMGSZ),
                       augment=TemporalAugment(seed=7, **AUGMENT),
                       letterbox=True, out_dtype="uint8", return_meta=True)

def show(ds, row_title, ax_row):
    x, tb, meta = ds[pick]
    frames = [x[0:3], x[3:6], x[6:9]]
    labels = ["t-1", "t   (+ ground truth)", "t+1"]
    for ax, f, lab in zip(ax_row, frames, labels):
        ax.imshow(f.permute(1, 2, 0).numpy())
        ax.set_title(f"{row_title} · {lab}", fontsize=10.5, fontweight="bold")
        ax.axis("off"); ax.grid(False)
    for b in tb.numpy():
        _, xc, yc, w, h = b
        ax_row[1].add_patch(Rectangle(((xc - w/2) * IMGSZ, (yc - h/2) * IMGSZ),
                                      w * IMGSZ, h * IMGSZ, fill=False,
                                      ec="#00d43f", lw=2.0))
        ax_row[1].add_patch(Rectangle(((xc - w/2) * IMGSZ - 20, (yc - h/2) * IMGSZ - 20),
                                      w * IMGSZ + 40, h * IMGSZ + 40, fill=False,
                                      ec="#00d43f", lw=.8, ls=":"))
    return tb, meta

fig, axes = plt.subplots(2, 3, figsize=(15, 9.4))
tb0, meta = show(clean, "letterboxed", axes[0])
tb1, _ = show(aug, "augmented", axes[1])
fig.suptitle(f"{meta['dataset']} / {meta['seq_key']}  frame {meta['frame_idx']} "
             f"({meta['modality']})", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

hero([
    ("[9, 640, 640]", "tensor delivered to the backbone", DS_COLOR["anti_uav"]),
    (f"{tb0.shape[0]}", "ground-truth boxes, clean", MUTED),
    (f"{tb1.shape[0]}", "after augmentation (copy-paste can add micro targets)", "#0f7a2e"),
    (f"{(tb0[:,3]*tb0[:,4]).min()*100:.4f}%" if len(tb0) else "—",
     "smallest target, as a share of the canvas", ACCENT),
])
panel("Top row is what evaluation sees; bottom row is what training sees. The <b>same</b> "
      "geometric and photometric transform is applied to all three frames &mdash; if it were "
      "not, the stack would encode motion the camera never saw, and the entire reason for "
      "feeding the network three frames would be poisoned. Note the grey letterbox bars: the "
      "aspect ratio is preserved, so a drone with a true W/H of 1.66 presents as 1.66 whether "
      "it came from a 640&times;512 thermal sensor or a 1920&times;1080 visible one.",
      DS_COLOR["cst"])

In [ ]:
from imbalance_sampler import hardness_from_index, hardness_to_weights, CATEGORIES

title("Hardness-aware sampling — what the model will actually see", kicker="Act III · sampling")

h = hardness_from_index(ctx["train_ds"].index)
w, cats = hardness_to_weights(h, SAMPLER_WEIGHTS)
uni = np.array([(cats == c).mean() for c in CATEGORIES])
wts = np.array([w[cats == c].sum() for c in CATEGORIES]); wts = wts / wts.sum()

fig, ax = plt.subplots(figsize=(10.5, 4.2))
y = np.arange(len(CATEGORIES)); hgt = .36
ax.barh(y + hgt/2, uni * 100, height=hgt, color="#c9c9c4", label="corpus (uniform)")
ax.barh(y - hgt/2, wts * 100, height=hgt, label="after hardness weighting",
        color=[ACCENT if c in ("bird", "micro") else DS_COLOR["anti_uav"] for c in CATEGORIES])
for i, c in enumerate(CATEGORIES):
    ax.text(max(uni[i], wts[i]) * 100 + .8, i, f"x{SAMPLER_WEIGHTS[c]:g}", va="center",
            fontsize=10, color=MUTED)
ax.set_yticks(y); ax.set_yticklabels([c.capitalize() for c in CATEGORIES])
ax.set_xlabel("share of what the model actually sees (%)")
ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
ax.set_title("Weighted sampling shifts exposure toward the failure modes",
             fontsize=13, fontweight="bold", loc="left")
plt.show()

hard_b = uni[CATEGORIES.index("bird")] + uni[CATEGORIES.index("micro")]
hard_a = wts[CATEGORIES.index("bird")] + wts[CATEGORIES.index("micro")]
hero([
    (f"{hard_b*100:.1f}%", "hard cases under uniform sampling", MUTED),
    (f"{hard_a*100:.1f}%", "hard cases after weighting", ACCENT),
    (f"{hard_a/max(hard_b,1e-9):.2f}x", "increase in hard-case exposure", "#0f7a2e"),
    (f"{SAMPLES_PER_EPOCH:,}", "weighted draws per epoch (the iteration budget)",
     DS_COLOR["anti_uav"]),
])

---
<div style="background:#f7f3fb;border-left:5px solid #6b4fa0;padding:14px 20px;border-radius:0 8px 8px 0;color:#0f1114">

# ACT IV · Training
### Staged, and detached — so the machine stays yours while it runs

</div>

**S0 first, always.** If a 4 M-parameter network cannot drive the loss to near zero on
*sixteen fixed images*, the bug is in the labels, the collate or the assigner — and no
amount of full-scale training will find it for you. It takes two minutes and it has
caught more broken pipelines than any other test in the roadmap. It now runs as the first
step of the background job, and a failure there aborts the staircase before S1 starts.

**Then the run leaves this kernel.** The cell below hands `overfit → smoke → baseline` to
a detached process and returns in about a second. That is not a convenience:

| | Training *inside* the kernel | Training *detached* |
|---|---|---|
| Survives closing the notebook | no | **yes** |
| Survives a kernel restart / VS Code restart | no | **yes** |
| Kernel's own RAM (torch, CUDA context, the sample index) | held for the whole run | **released before the run starts** |
| CPU priority | same as the UI you are typing into | **below normal — the foreground wins** |
| A cell you accidentally run mid-training | queues behind the run | harmless |

The GPU is *not* throttled by any of this. CPU priority and GPU scheduling are separate
resources: the card keeps running our kernels flat out while Windows simply stops letting
the loader's JPEG decoding preempt whatever is redrawing a slide.


In [ ]:
title("Hand the run to a detached process", kicker="Act IV &middot; launch")

# The job this kernel built for the pre-flight goes FIRST, and it has to go
# before anything else is created. Its loader still owns live worker processes
# (persistent_workers=True), and each of those holds a copy of the sample index
# and its prefetch queue. Launching the real run on top of that means two
# complete corpora and two sets of workers resident at once -- which is exactly
# how a 24 GB machine ends up having a worker killed by the OS.
T.release(trainer, ctx, free_manifest=True)
trainer = ctx = None
gc.collect()
torch.cuda.empty_cache()

STAGE_OVERRIDES = {
    "smoke":    dict(batch=BATCH, imgsz=IMGSZ, lr0=LR0),
    "pilot":    dict(batch=BATCH, imgsz=IMGSZ, lr0=LR0,
                     samples_per_epoch=min(SAMPLES_PER_EPOCH, 15_000)),
    "baseline": dict(batch=BATCH, imgsz=IMGSZ, lr0=LR0, epochs=EPOCHS,
                     samples_per_epoch=SAMPLES_PER_EPOCH, patience=PATIENCE),
}

if BACKGROUND:
    bg = T.launch_background(
        RUN_STAGES,
        batch=BATCH, workers=WORKERS, epochs=EPOCHS,
        samples_per_epoch=SAMPLES_PER_EPOCH, lr0=LR0, patience=PATIENCE,
        imgsz=IMGSZ,
        augment=AUGMENT, sampler_weights=SAMPLER_WEIGHTS,
        temporal_gap=TEMPORAL_GAP,
        reserve_gb=RESERVE_RAM_GB, priority=BG_PRIORITY,
        # The pre-flight above ran on this exact configuration, so the detached
        # process does not pay for it a second time.
        preflight=not PREFLIGHT_OK,
        resume=RESUME_FROM, tag=BG_TAG)

    hero([
        (str(bg["pid"]), "process id &mdash; <b>detached</b>, not a child of this kernel",
         DS_COLOR["anti_uav"]),
        (" &rarr; ".join(bg["stages"]), "stages, run back to back", DS_COLOR["cst"]),
        (BG_PRIORITY.replace("_", " "), "CPU priority &mdash; GPU share unchanged", MUTED),
        (f"{RESERVE_RAM_GB:.0f} GB", "host RAM left to you", ACCENT),
    ], caption=f"log: <code>{bg['log']}</code>")

    panel("<b>You can stop here.</b> The run is no longer attached to anything in this "
          "window &mdash; close the notebook, restart the kernel, close VS Code, and it "
          "keeps going. Doing so is in fact the <i>recommended</i> next step: this "
          "kernel is still holding torch, a CUDA context and matplotlib, which is a "
          "couple of GB you could be spending on a slide deck.<br><br>"
          "To check on it later, from anywhere:<br>"
          "&bull; re-run the monitor cell below (it reads files, not memory, so it "
          "works after a kernel restart), or<br>"
          "&bull; <code>venv\\Scripts\\python c_model\\train.py --status</code> in a "
          "terminal, or<br>"
          "&bull; watch the GPU with <code>nvidia-smi dmon -s pucm</code>.<br><br>"
          "To stop it: <code>venv\\Scripts\\python c_model\\train.py --stop</code>. "
          "The last completed epoch is already in <code>weights/last.pt</code> with its "
          "optimiser and EMA state, so nothing is lost &mdash; set "
          "<code>RESUME_FROM</code> and relaunch.", "#0f7a2e")
else:
    panel("<b>BACKGROUND is False</b> &mdash; training will run inside this kernel and "
          "this cell will not return for hours. The kernel keeps the trainer, both "
          "loaders and their worker processes resident the whole time, and closing the "
          "notebook kills the run. Set <code>BACKGROUND = True</code> unless you are "
          "deliberately debugging the loop.", ACCENT)
    live_results = T.run_stages(
        RUN_STAGES, device=device, stage_overrides=STAGE_OVERRIDES,
        augment_kwargs=AUGMENT, sampler_weights=SAMPLER_WEIGHTS,
        temporal_gap=TEMPORAL_GAP, preflight=not PREFLIGHT_OK,
        resume_from=RESUME_FROM)
    print("\nfinished:", list(live_results))


In [ ]:
# [TEST ONLY] block until the detached run finishes, so every cell below
# executes its real path instead of its "still training" guard.
import time as _t
_deadline = _t.time() + 45 * 60
while T.bg_status(BG_TAG, verbose=False).get("alive", False) and _t.time() < _deadline:
    _t.sleep(15)
print("background run finished:",
      not T.bg_status(BG_TAG, verbose=False).get("alive", False))


In [ ]:
import pandas as pd

# Deliberately self-sufficient: this cell reads FILES, never the launcher's
# variables, so it works after a kernel restart, from a second notebook, or days
# later. Re-run it whenever you want to know how the run is doing.
if "T" not in dir():
    purge_project_modules()
    import train as T

title("How is it going?", kicker="Act IV &middot; monitor")

status = T.bg_status(BG_TAG, lines=18, verbose=False)


def stage_results(stage):
    """Rebuild one stage's result record from disk (results.csv + args.json)."""
    name = (status.get("run_names") or {}).get(stage) or T.STAGES.get(stage, {}).get("name")
    if not name:
        return None
    run_dir = RUNS / name
    csv_p = run_dir / "results.csv"
    if not csv_p.exists():
        return None
    df = pd.read_csv(csv_p)
    if df.empty:
        return None
    hist = df.to_dict("records")
    best_fit, best_ep = float("nan"), -1
    if "fitness" in df and df["fitness"].notna().any():
        i = df["fitness"].idxmax()
        best_fit, best_ep = float(df.at[i, "fitness"]), int(df.at[i, "epoch"])
    return {"run_dir": str(run_dir),
            "best": str(run_dir / "weights" / "best.pt"),
            "best_fitness": best_fit, "best_epoch": best_ep,
            "history": hist, "epochs": len(df)}


# `results` keeps the shape the evaluation cells below already expect, so
# nothing downstream cares whether training ran here or in another process.
results = {s: r for s in RUN_STAGES if (r := stage_results(s)) is not None}

alive = status.get("alive", False)
if not status.get("exists"):
    print("No background run has been launched yet with tag", repr(BG_TAG))
else:
    _rows = []
    for s in RUN_STAGES:
        r = results.get(s)
        _rows.append([s,
                      "running" if alive else "not running",
                      "-" if not r else f"{r['epochs']}",
                      "-" if not r or r["best_epoch"] < 0 else f"{r['best_fitness']:.4f}",
                      "-" if not r or r["best_epoch"] < 0 else f"{r['best_epoch']}"])
    table(pd.DataFrame(_rows, columns=["Stage", "Process", "Epochs recorded",
                                       "Best fitness", "Best epoch"]),
          note=f"Process {status.get('pid')} is "
               f"<b>{'RUNNING' if alive else 'not running (finished, stopped, or failed)'}</b>. "
               f"Progress is read from <code>results.csv</code>, which the trainer "
               f"rewrites after every epoch &mdash; a file, not a socket, so this is "
               f"readable from any process at any time.")

    print("-" * 78)
    print(f"tail of {status.get('log')}")
    print("-" * 78)
    for ln in status.get("tail", []):
        print(ln)

    if not alive and not results:
        panel("<b>The process is gone and no epoch was recorded.</b> Read the log tail "
              "above: an S0 failure, a failed pre-flight check or an environment "
              "problem all abort before the first epoch, and all of them print the "
              "reason.", ACCENT)


In [ ]:
title("Training curves", kicker="Act IV &middot; what happened")

last = [s for s in ("baseline", "pilot", "smoke") if s in results]
if last:
    stage = last[0]
    h = results[stage]["history"]
    ep = [r["epoch"] for r in h]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
    for k, c in (("box_loss", DS_COLOR["anti_uav"]), ("cls_loss", DS_COLOR["cst"]),
                 ("dfl_loss", DS_COLOR["det_fly"])):
        ax[0].plot(ep, [r[k] for r in h], label=k, color=c, lw=2)
    ax[0].set_title("training loss", fontweight="bold", loc="left"); ax[0].legend()
    # Epochs without a validation pass leave those columns blank in results.csv,
    # which pandas reads back as NaN -- `is not None` would let them through and
    # plot a line full of holes.
    v = [(r["epoch"], r.get("mAP50"), r.get("mAP50-95"), r.get("micro_AP50")) for r in h
         if pd.notna(r.get("mAP50"))]
    if v:
        e2 = [x[0] for x in v]
        ax[1].plot(e2, [x[1] for x in v], label="mAP@50", color=DS_COLOR["anti_uav"], lw=2)
        ax[1].plot(e2, [x[2] for x in v], label="mAP@50-95", color=ACCENT, lw=2)
        ax[1].plot(e2, [x[3] for x in v], label="micro AP@50", color="#0f7a2e", lw=2.6)
        ax[1].set_title("validation — micro AP is the thesis", fontweight="bold", loc="left")
        ax[1].legend()
    ax[2].plot(ep, [r["lr"] for r in h], color="#6b4fa0", lw=2)
    ax[2].set_title("learning rate", fontweight="bold", loc="left")
    for a in ax:
        a.set_xlabel("epoch")
    fig.tight_layout(); plt.show()

    _bf = results[stage]["best_fitness"]
    hero([
        ("—" if pd.isna(_bf) else f"{_bf:.4f}", "best fitness", DS_COLOR["anti_uav"]),
        (f"{results[stage]['best_epoch']}", "epoch it was reached", MUTED),
        (f"{sum(r['epoch_time_s'] for r in h)/3600:.2f} h", "wall-clock training time", MUTED),
        (f"{np.mean([r['img_per_s'] for r in h]):.0f}", "images/s sustained", DS_COLOR["cst"]),
    ], caption="fitness = 0.30&middot;mAP@50 + 0.35&middot;mAP@50-95 + 0.35&middot;micro-AP@50 "
               "&mdash; model selection is weighted toward the targets the project exists "
               "to detect. These curves are as of the last completed epoch; re-run this "
               "cell to refresh them while the background run continues.")
else:
    print("no epoch has been recorded yet — re-run the monitor cell above in a "
          "few minutes, then this one.")


---
<div style="background:#f4f4f2;border-left:5px solid #1a1a1a;padding:14px 20px;border-radius:0 8px 8px 0;color:#0f1114">

# ACT V · Evaluation
### The breakdown that can actually falsify the thesis

</div>

Aggregate mAP is the wrong headline for this project. It is dominated by large, centred,
easy targets — exactly the population Review 1 argued is *not* the hard part. A model that
finds every 40 px gimbal-centred drone and no 9 px ones scores well and is useless.

So the report below leads with **AP by target size**, and adds the four numbers that
correspond to specific, named failure modes: cross-dataset gap (centre bias), IR vs RGB
(modality balance), bird false positives (the classic counter-UAS false alarm), and
detections on `exist = 0` frames (hallucinating through an occlusion).

In [ ]:
import evaluation as E
from stacked_dataset import build_split_samples, load_label_cache, aerotrack_collate
from torch.utils.data import DataLoader

title("Full evaluation on the held-out validation sequences", kicker="Act V &middot; results")

# Evaluation loads a checkpoint onto the same 8 GB card the background run is
# using, and runs its own loader workers on the same RAM. Doing that mid-run is
# how you turn a training job into an OOM. So: only when the run is finished.
TRAINING_BUSY = T.bg_status(BG_TAG, verbose=False).get("alive", False)

BEST = None
for s in ("baseline", "pilot", "smoke"):
    if s in results and Path(results[s]["best"]).exists():
        BEST = Path(results[s]["best"]); break
if BEST is None:
    cand = sorted(RUNS.glob("*/weights/best.pt"), key=lambda p: p.stat().st_mtime)
    BEST = cand[-1] if cand else None

rep = None
if TRAINING_BUSY:
    panel("<b>Training is still running &mdash; evaluation skipped.</b> This cell would "
          "load a second model onto the same 8&nbsp;GB card and start its own loader "
          "workers on the same RAM, which is a good way to turn a healthy run into an "
          "OOM. Come back when the monitor cell reports the process is no longer "
          "running, then run this cell and the ones below.", ACCENT)
elif EVAL_AFTER_TRAINING and BEST:
    from aerotrack_trainer import load_checkpoint
    model, ck = load_checkpoint(BEST, device, prefer_ema=True)
    _fit = ck.get("best_fitness")
    print(f"[eval] {BEST}  (epoch {ck.get('epoch')}, "
          f"fitness {_fit:.4f})" if isinstance(_fit, float) else f"[eval] {BEST}")

    val_s = build_split_samples(UNIFIED / "manifest.csv", UNIFIED / "splits.json", "val",
                                temporal_gap=max(TEMPORAL_GAP, 1))
    if EVAL_CAP_PER_DATASET:
        val_s = T.subsample(val_s, {d: EVAL_CAP_PER_DATASET
                                    for d in {s["dataset"] for s in val_s}})
    val_ds = AeroTrackDataset(val_s, img_size=(IMGSZ, IMGSZ), augment=None,
                              letterbox=True, out_dtype="uint8", return_meta=True,
                              label_cache=load_label_cache(UNIFIED / "label_cache.npz"))
    EVAL_WORKERS = T.plan_workers(WORKERS, verbose=True)
    dl_kw = dict(num_workers=EVAL_WORKERS, pin_memory=True,
                 collate_fn=aerotrack_collate, worker_init_fn=T._worker_init)
    val_dl = DataLoader(val_ds, batch_size=max(BATCH, 8), shuffle=False, **dl_kw)
    t0 = time.time()
    recs = E.collect_records(model, val_dl, device, imgsz=IMGSZ,
                             amp_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else None)
    rep = E.full_report(recs, save_dir=BEST.parent.parent / "eval_val")
    print(f"[eval] {len(recs):,} images in {time.time()-t0:.0f}s")
    E.print_report(rep)
else:
    print("evaluation skipped (EVAL_AFTER_TRAINING is False, or no checkpoint found)")


In [ ]:
if rep:
    title("The headline: AP by target size", kicker="Act V · the SPD-Conv test")

    bs = rep["by_size"]
    fmt = lambda v: ("—" if v is None else f"{v:.3f}")
    hero([
        (fmt(bs['micro']['map50']), f"<b>micro</b> AP@50 &mdash; targets &lt;0.03% of frame "
                                        f"({bs['micro']['n_gt']:,} boxes)", ACCENT),
        (fmt(bs['small']['map50']), f"small AP@50 &mdash; 0.03&ndash;1% ({bs['small']['n_gt']:,})",
         DS_COLOR["cst"]),
        (fmt(bs['standard']['map50']), f"standard AP@50 &mdash; &gt;1% ({bs['standard']['n_gt']:,})",
         DS_COLOR["anti_uav"]),
        (fmt(rep['overall']['map50']), "aggregate mAP@50 <i>(the least informative number here)</i>",
         MUTED),
    ], caption="If SPD-Conv earns its place, it earns it in the first column. Run the "
               "<code>no_spd</code> ablation to put a control next to it.")

    rows = []
    for d, v in rep["by_dataset"].items():
        rows.append([d, f"{v['n_images']:,}",
                     "—" if v["map50"] is None else f"{v['map50']:.4f}",
                     "—" if v["map5095"] is None else f"{v['map5095']:.4f}",
                     "—" if v["micro_ap50"] is None else f"{v['micro_ap50']:.4f}"])
    table(pd.DataFrame(rows, columns=["Dataset", "Images", "mAP@50", "mAP@50-95", "micro AP@50"]),
          note="<b>Read the Anti-UAV vs CST gap as a bias measurement, not a difficulty one.</b> "
               "Anti-UAV is 90% centre-concentrated because it is gimbal-tracked footage &mdash; "
               "the camera actively keeps the drone in the middle. A large gap here means the "
               "model learned &lsquo;look in the centre&rsquo; and will fail on a fixed camera.")

    if rep["by_modality"]:
        rows = [[m, f"{v['n_images']:,}",
                 "—" if v["map50"] is None else f"{v['map50']:.4f}",
                 "—" if v["map5095"] is None else f"{v['map5095']:.4f}"]
                for m, v in rep["by_modality"].items()]
        table(pd.DataFrame(rows, columns=["Modality", "Images", "mAP@50", "mAP@50-95"]),
              note="Anti-UAV ships every sequence twice &mdash; once thermal, once visible, "
                   "same scene, same instant. This is therefore a <b>confound-free</b> modality "
                   "comparison. Review 1 measured 43.4 vs 22.0 target&ndash;background contrast in "
                   "IR's favour; if RGB lags far behind here, BatchNorm is seeing two very "
                   "different input distributions and modality-balanced batches are the fix.")

    b, e = rep["bird_fp"], rep["empty_frame_fp"]
    hero([
        (f"{b['fp_per_image']:.3f}" if b["fp_per_image"] is not None else "—",
         f"false positives per <b>bird-only</b> image ({b['images']:,} imgs)", DS_COLOR["det_fly"]),
        (f"{e['fp_per_image']:.3f}" if e["fp_per_image"] is not None else "—",
         f"FP per <b>empty</b> frame &mdash; exist=0 ({e['images']:,} imgs)", ACCENT),
        (f"{rep['overall']['recall'].get('uav_drone', 0):.3f}", "drone recall at best-F1", "#0f7a2e"),
        (f"{rep['overall']['precision'].get('uav_drone', 0):.3f}", "drone precision at best-F1",
         DS_COLOR["anti_uav"]),
    ], caption=f"Both FP rates are measured at confidence {b['conf_thres']}. A bird is the single "
               f"most likely false positive for any counter-UAS system, which is exactly why the "
               f"Det-Fly bird class was kept as a hard negative and over-drawn by the sampler.")

    display(HTML(f"<div style='{_BOX}padding:14px'><b>Saved artifacts</b><br>"
                 f"<code>{BEST.parent.parent / 'eval_val'}</code> &mdash; "
                 f"evaluation_report.json, PR_curve.png, AP_by_size.png</div>"))

---
## ACT VI · Ablations — each one defends a decision

Every ablation below removes exactly one component and changes nothing else. They are run
at reduced epochs on purpose: what is needed is the **direction and size** of the effect,
not a second set of final numbers.

| Ablation | What it removes | What should happen if the Review-1 argument is right |
|---|---|---|
| `no_spd` | SPD-Conv → stock stride-2 Conv | **micro AP falls sharply**; aggregate mAP barely moves |
| `single_frame` | temporal stack (t−1=t=t+1) | loss on fast-motion and blurred targets |
| `no_sampler` | hardness weighting | worse micro and bird recall |
| `no_letterbox` | aspect-preserving resize | worse cross-dataset consistency |
| `scratch` | COCO transfer | slower convergence, similar final |
| `p2_head` | *adds* a P2/4 scale | micro AP gain, at a speed cost |
| `no_augment` | all augmentation | earlier overfitting |

In [ ]:
title("Ablations", kicker="Act VI &middot; defending the design")

ablation_results = {}
if RUN_ABLATIONS and T.bg_status(BG_TAG, verbose=False).get("alive", False):
    panel("<b>Ablations skipped &mdash; the baseline is still training.</b> Two training "
          "jobs on one 8&nbsp;GB card is an OOM, not a speedup. Ablations are also the "
          "one part of this notebook that is better run from a terminal, because each is "
          "a multi-hour job of its own:<br><br>"
          "<code>venv\\Scripts\\python c_model\\train.py --stage baseline "
          "--ablation no_spd --epochs 40 --background --tag abl_no_spd</code><br><br>"
          "The <code>--tag</code> keeps each one's status file separate, so several can "
          "be queued and tracked independently.", ACCENT)
else:
    for abl in RUN_ABLATIONS:
        print("\n" + "#" * 78)
        print(f"#  ABLATION: {abl}")
        print("#" * 78, flush=True)
        tr, cx = T.build("baseline", ablation=abl, device=device,
                         overrides=dict(workers=WORKERS, batch=BATCH, imgsz=IMGSZ,
                                        lr0=LR0, epochs=ABLATION_EPOCHS,
                                        samples_per_epoch=SAMPLES_PER_EPOCH, patience=0),
                         augment_kwargs=AUGMENT, sampler_weights=SAMPLER_WEIGHTS)
        tr.train()
        ablation_results[abl] = {"best_fitness": tr.best_fitness,
                                 "run_dir": str(tr.save_dir),
                                 "best": str(tr.save_dir / "weights" / "best.pt")}
        T.release(tr, cx, free_manifest=True)
        del tr, cx; gc.collect(); torch.cuda.empty_cache()

if ablation_results:
    rows = [[k, f"{v['best_fitness']:.4f}", v["run_dir"]]
            for k, v in ablation_results.items()]
    table(pd.DataFrame(rows, columns=["Ablation", "Best fitness", "Run directory"]),
          note="Run <code>c_model/val.py --weights &lt;run&gt;/weights/best.pt</code> on each to "
               "get its full size-stratified breakdown, then compare the <b>micro</b> column "
               "against the baseline. That comparison, not the aggregate mAP, is the ablation.")
elif not RUN_ABLATIONS:
    print("no ablations requested (RUN_ABLATIONS is empty)")


---
## ACT VII · The test split — spend it once

`cst[test]` is 72,175 frames of the only footage in this corpus that is **not**
gimbal-tracked. It is the single measurement that can tell you whether the model
generalises off centre-biased data, and it is worth exactly one use. Every time it informs
a choice — a checkpoint, an epoch count, a threshold — it stops being a test set.

`RUN_FINAL_TEST` is left `True` because this run's baseline *is* the final
model — but the cell below refuses to spend the split while training is still
in flight, and it is the last thing you should run, after every other decision
has already been made.

In [ ]:
if RUN_FINAL_TEST and T.bg_status(BG_TAG, verbose=False).get("alive", False):
    panel("<b>The test split was not touched &mdash; training is still running.</b> "
          "<code>cst[test]</code> is worth exactly one use, and it is only worth that "
          "on a <i>finished</i> model. Re-run this cell once the monitor reports the "
          "process has stopped.", ACCENT)
elif RUN_FINAL_TEST and BEST:
    title("Final test evaluation", kicker="Act VII &middot; once, and only once")
    if "model" not in dir():                      # evaluation cell was skipped
        from aerotrack_trainer import load_checkpoint
        model, ck = load_checkpoint(BEST, device, prefer_ema=True)
    test_s = build_split_samples(UNIFIED / "manifest.csv", UNIFIED / "splits.json", "test",
                                 temporal_gap=max(TEMPORAL_GAP, 1))
    if EVAL_CAP_PER_DATASET:
        test_s = T.subsample(test_s, {d: EVAL_CAP_PER_DATASET
                                      for d in {s["dataset"] for s in test_s}})
    test_ds = AeroTrackDataset(test_s, img_size=(IMGSZ, IMGSZ), augment=None,
                               letterbox=True, out_dtype="uint8", return_meta=True,
                               label_cache=load_label_cache(UNIFIED / "label_cache.npz"))
    test_dl = DataLoader(test_ds, batch_size=max(BATCH, 8), shuffle=False,
                         num_workers=T.plan_workers(WORKERS, verbose=False),
                         pin_memory=True, collate_fn=aerotrack_collate,
                         worker_init_fn=T._worker_init)
    trecs = E.collect_records(model, test_dl, device, imgsz=IMGSZ,
                              amp_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else None)
    trep = E.full_report(trecs, save_dir=BEST.parent.parent / "eval_test")
    E.print_report(trep)

    cst = trep["by_dataset"].get("cst")
    au = trep["by_dataset"].get("anti_uav")
    if cst and au:
        gap = (au["map50"] or 0) - (cst["map50"] or 0)
        panel(f"<b>Centre-bias verdict.</b> Anti-UAV mAP@50 = {au['map50']:.4f}, "
              f"CST mAP@50 = {cst['map50']:.4f}, gap = <b>{gap:+.4f}</b>. Anti-UAV is "
              f"gimbal-tracked and 90% centre-concentrated; CST is not. A large positive gap "
              f"means the model leaned on target position rather than target appearance.",
              ACCENT if gap > 0.15 else "#0f7a2e")
else:
    print("test split not evaluated — RUN_FINAL_TEST is False, or no checkpoint exists yet.")


---
## Where this goes next

| Next | Command |
|---|---|
| Check on the background run | `python c_model/train.py --status` |
| Stop it (last.pt keeps optimiser + EMA state) | `python c_model/train.py --stop` |
| Restart it where it left off | set `RESUME_FROM = "runs/aerotrack_spd_v1/weights/last.pt"`, re-run |
| Watch the GPU while it runs | `nvidia-smi dmon -s pucm` |
| Full evaluation on any checkpoint | `python c_model/val.py --weights runs/<name>/weights/best.pt` |
| Cross-domain probe (once) | `python c_model/val.py --split test --confirm-test --datasets cst` |
| One ablation, also detached | `python c_model/train.py --stage baseline --ablation no_spd --epochs 40 --background --tag abl_no_spd` |
| Annotated demo video + latency HUD | `python c_model/inference.py --source clip.mp4 --weights runs/<name>/weights/best.pt` |
| ByteTrack association layer | detections + the `exist=0` flags already preserved as 0-byte labels |
| Social-LSTM trajectory forecaster | `a_inspection/artifacts/velocities.pkl` — 706,147 transitions, ready |
| TensorRT FP16 export | verify ONNX handles SPDConv's slicing ops, or write a plugin |

**Limitations, stated rather than hidden**

1. **1080p downscaling.** The RGB stream still loses ~3× linear resolution at 640×640.
   Letterboxing fixed the *aspect* distortion, not the *resolution* loss; tiling is the fix.
2. **Centre bias.** Mitigated by translation augmentation, hardness weighting and CST
   validation — but only the CST-test number in Act VII actually settles it.
3. **CST truncation.** The archive's final volume was never delivered; 163,275 files were
   recovered and everything past `train/urban-areas_1/000657.jpg` is unrecoverable. All CST
   figures are over the recovered subset, and we say so.
4. **Validation is scored in letterbox space.** Boxes are compared on the 640×640 canvas
   rather than at native resolution. This is self-consistent and standard, but it is not
   identical to a native-resolution COCO evaluation, and the difference is worth a footnote.
5. **The batch size is sized for a shared card.** `RESERVE_VRAM_GB` holds 2.2 GB back for
   the desktop, which costs some throughput against a dedicated machine. That is the
   deliberate price of being able to work while the model trains.

---

<div style="text-align:center;background:#f1f1ec;border:1px solid #c4c4be;border-radius:8px;
            color:#3f444c;font-size:12.5px;padding:14px 18px;margin-top:8px">
AeroTrack-Net · Review 2 · every table and figure above is generated live from
<code>b_pipeline/data_unified/</code> and <code>runs/</code>. Nothing is pre-rendered.
</div>
